In [ ]:
import sys
sys.path.append('/scratch/gilbreth/abelde/Thesis/StructureAwareGen/scripts/SEG-RDM')

sys.argv = [
    'rdm.main_rdm',
    '--config', 'rdm/configs/rdm_default.yaml',
    '--batch_size', '8',
    '--input_size', '256',
    '--epochs', '1',
    '--blr', '1e-6',
    '--weight_decay', '0.01',
    '--output_dir', '/tmp/seg_rdm_out',
    '--data_path', '/path/to/imagenet_or_dummy',
]

from rdm.main_rdm import get_args_parser, main
args = get_args_parser().parse_args()
main(args)

In [2]:
import os
import sys
import importlib

# -----------------------------
# Paths (edit if needed)
# -----------------------------
repo_root = "/scratch/gilbreth/abelde/Thesis/StructureAwareGen/scripts/SEG-RDM/"
config_path = "/scratch/gilbreth/abelde/Thesis/StructureAwareGen/scripts/SEG-RDM/rdm/configs/rdm_default.yaml"  # relative to repo_root is fine
output_dir = os.environ.get("OUTPUT_DIR", "/scratch/gilbreth/abelde/Thesis/StructureAwareGen/scripts/SEG-RDM/rdm/rdm_debug_out")
imagenet_dir = os.environ.get("IMAGENET_DIR", "/scratch/gilbreth/abelde/Thesis/StructureAwareGen/dataset/imagenet-1K-hf/")
# imagenet_dir = os.environ.get("IMAGENET_DIR", "/scratch/gilbreth/abelde/Thesis/StructureAwareGen/dataset/val2017/")

import os

# --- Make env:// rendezvous happy (single process) ---
os.environ["MASTER_ADDR"] = "127.0.0.1"
os.environ["MASTER_PORT"] = "29500"
os.environ["WORLD_SIZE"] = "1"
os.environ["RANK"] = "0"
os.environ["LOCAL_RANK"] = "0"


# -----------------------------
# Put repo on import path
# -----------------------------
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)



# If you already imported these in the notebook, reload to pick up edits/breakpoints
import main_rdm
importlib.reload(main_rdm)


sys.argv = [
    'rdm.main_rdm',
    '--config', config_path,
    '--input_size', '256',
    '--blr', '1e-6',
    '--weight_decay', '0.01',
    '--output_dir', output_dir,
    '--data_path', imagenet_dir,

    '--debug',
    '--epochs', '200',
    '--batch_size', '2',
    '--debug_n', '100',
    '--debug_steps', '5',
]




from rdm.main_rdm import get_args_parser, main
args = get_args_parser().parse_args()
# --- Make sure your args reflect single-process ---
args.world_size = 1
args.rank = 0
args.gpu = 0          # or set to your intended GPU index
args.dist_url = "env://"
main(args)


2026-01-30 17:27:26.185039: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1769812046.315402 3314531 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1769812046.353580 3314531 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1769812046.652511 3314531 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1769812046.652542 3314531 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1769812046.652544 3314531 computation_placer.cc:177] computation placer alr

KeyboardInterrupt: 

In [2]:
import os
import sys
import importlib

# -----------------------------
# Paths (edit if needed)
# -----------------------------
repo_root = "/scratch/gilbreth/abelde/Thesis/StructureAwareGen/scripts/SEG-RDM/"
config_path = "/scratch/gilbreth/abelde/Thesis/StructureAwareGen/scripts/SEG-RDM/rdm/configs/unified_seg_rdm.yaml"
output_dir = os.environ.get("OUTPUT_DIR", "/scratch/gilbreth/abelde/Thesis/StructureAwareGen/scripts/SEG-RDM/checkpoints/unified_seg_rdm_debug")
imagenet_dir = os.environ.get("IMAGENET_DIR", "/scratch/gilbreth/abelde/Thesis/StructureAwareGen/dataset/val2017/train")
sam_npz_dir = os.environ.get("SAM_NPZ_DIR", "/scratch/gilbreth/abelde/Thesis/StructureAwareGen/dataset/sam_embeddings/")

# --- Make env:// rendezvous happy (single process) ---
os.environ["MASTER_ADDR"] = "127.0.0.1"
os.environ["MASTER_PORT"] = "29500"
os.environ["WORLD_SIZE"] = "1"
os.environ["RANK"] = "0"
os.environ["LOCAL_RANK"] = "0"

# -----------------------------
# Put repo on import path
# -----------------------------
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

# Import the training module
sys.path.insert(0, os.path.join(repo_root, '../../'))  # Go up to scripts/SEG-RDM/
import train_unified_seg_rdm
importlib.reload(train_unified_seg_rdm)

# -----------------------------
# Create a minimal trainer for debugging
# -----------------------------
from train_unified_seg_rdm import UnifiedSegRDMTrainer
import yaml

# Load and modify config for debugging
with open(config_path, 'r') as f:
    config = yaml.safe_load(f)

# Override config with debug settings
config['data']['params']['image_dir'] = imagenet_dir
config['data']['params']['mask_npz_dir'] = sam_npz_dir
config['data']['params']['batch_size'] = 2  # Small batch for debugging
config['data']['params']['num_workers'] = 0  # No multiprocessing for easier debugging

config['training']['max_steps'] = 50  # Just a few steps for testing
config['training']['log_interval'] = 5
config['training']['checkpoint_interval'] = 25
config['training']['use_fp16'] = False  # Disable FP16 for clearer debugging

# Update output directory
config['training']['checkpoint_dir'] = output_dir
config['logging']['tensorboard_dir'] = os.path.join(output_dir, 'tensorboard')

# Save modified config temporarily
debug_config_path = os.path.join(output_dir, 'debug_config.yaml')
os.makedirs(output_dir, exist_ok=True)
with open(debug_config_path, 'w') as f:
    yaml.dump(config, f)

print(f"Debug config saved to: {debug_config_path}")
print(f"Output directory: {output_dir}")
print(f"Image directory: {imagenet_dir}")
print(f"SAM NPZ directory: {sam_npz_dir}")
print("\nStarting Unified Segmentation RDM training...")
print("=" * 80)

# Create trainer and start training
# You can set breakpoints in train_unified_seg_rdm.py and debug from here
trainer = UnifiedSegRDMTrainer(config_path=debug_config_path, resume_from=None)
trainer.train()


Debug config saved to: /scratch/gilbreth/abelde/Thesis/StructureAwareGen/scripts/SEG-RDM/checkpoints/unified_seg_rdm_debug/debug_config.yaml
Output directory: /scratch/gilbreth/abelde/Thesis/StructureAwareGen/scripts/SEG-RDM/checkpoints/unified_seg_rdm_debug
Image directory: /scratch/gilbreth/abelde/Thesis/StructureAwareGen/dataset/val2017/train
SAM NPZ directory: /scratch/gilbreth/abelde/Thesis/StructureAwareGen/dataset/sam_embeddings/

Starting Unified Segmentation RDM training...
Loaded config from: /scratch/gilbreth/abelde/Thesis/StructureAwareGen/scripts/SEG-RDM/checkpoints/unified_seg_rdm_debug/debug_config.yaml
data:
  batch_size: 32
  num_workers: 8
  params:
    batch_size: 2
    file_ext: '*.jpg'
    image_dir: /scratch/gilbreth/abelde/Thesis/StructureAwareGen/dataset/val2017/train
    image_size: 256
    mask_npz_dir: /scratch/gilbreth/abelde/Thesis/StructureAwareGen/dataset/sam_embeddings/
    max_segments: 250
    normalize: true
    num_workers: 0
  pin_memory: true
  shu

/home/abelde/.local/lib/python3.9/site-packages/torch/utils/data/dataloader.py:626: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 5, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(
wandb: Currently logged in as: abhinaybelde99 (abhinaybelde99-purdue-university-fort-wayne) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Weights & Biases logging enabled


2026-01-30 17:45:42.042262: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1769813142.063295 3330912 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1769813142.069515 3330912 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1769813142.085791 3330912 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1769813142.085810 3330912 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1769813142.085824 3330912 computation_placer.cc:177] computation placer alr

TensorBoard logging enabled

=== Starting Training ===


Training:   0%|          | 0/50 [00:00<?, ?it/s]/home/abelde/.local/lib/python3.9/site-packages/torch/utils/data/dataloader.py:626: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 5, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(


RuntimeError: The size of tensor a (256) must match the size of tensor b (3) at non-singleton dimension 1